Stencil system

Install the stencil_lib wheel using pip 

In [30]:
import subprocess, sys, pathlib

# location of .whl file
_base = (
    pathlib.Path(__file__).resolve().parent
    if "__file__" in globals()
    else pathlib.Path.cwd()
 )
_search_paths = [
    _base,
    _base / "build" / "dist",
    _base / ".." / "build" / "dist",
    _base / ".." / ".." / "build" / "dist",
]
_wheel = next(
    (str(w) for p in _search_paths for w in p.glob("stencil_lib-*.whl")),
    None
)
if _wheel is None:
    raise FileNotFoundError("stencil_lib wheel not found. Run 'inv build' or place the wheel alongside the notebook.")

# pip install the wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       _wheel, "--force-reinstall"])

print(f"stencil_lib installed from: {_wheel}")


stencil_lib installed from: d:\Adithya\Knot_theory_project\repo\SecureCameraZoom\stencil_system\python\..\..\build\dist\stencil_lib-0.1.0-py3-none-any.whl


Start using the stencil_lib library

In [31]:
# Start using the library
from stencil_lib import CipherConfig

In [32]:
class AESConfig(CipherConfig):
    algo = "aes"
    def __init__(self, key: bytes, mode: int, **mode_params):
        self.parameters = {
            "key": key,
            "mode": mode,
            "mode_params": mode_params
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.encrypt(plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.decrypt(ciphertext)


In [33]:
class CaesarConfig(CipherConfig):
    algo = "caesar"
    def __init__(self, shift: int):
        self.parameters = {
            "shift": shift
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b + shift) % 256 for b in plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b - shift) % 256 for b in ciphertext)


In [34]:
class VigenereConfig(CipherConfig):
    algo = "vigenere"
    def __init__(self, keyword: bytes):
        self.parameters = {
            "keyword": keyword
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b + key[i % key_len]) % 256 for i, b in enumerate(plaintext))
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b - key[i % key_len]) % 256 for i, b in enumerate(ciphertext))


# Stencil_system demo

The stencil library renders 3 APIs namely keygen, encrypt and decrypt

The usage of these APIs are demonstarated below.

In [35]:
import random
import stencil_lib


grid_shape = stencil_lib.GridShape(n=2, shape=(12, 12))
in_byte_len = 16
plaintext = random.randbytes(in_byte_len)
num_partitions = 4
print(f"Plaintext : {' '.join(f'{b:02x}' for b in plaintext)}")

# cipher_cfg = AESConfig(key=random.randbytes(16), mode=1, iv=random.randbytes(16))
# cipher_cfg = CaesarConfig(shift=13)
cipher_cfg = VigenereConfig(keyword=b"KEY")

# --- New: create StencilConfig ---
stencil_cfg = stencil_lib.StencilConfig(
    total_bytes=in_byte_len,
    num_partitions=num_partitions,
    grid_shape=grid_shape,
)

Plaintext : 4b 09 f4 dc 49 af 75 48 f7 62 33 98 61 e8 64 18


In [36]:
# Grid generation is optional. If Grid is none, 
# it is randomly generated internally inside encrypt API.
grid = stencil_lib.generate_random_grid(stencil_cfg.grid_shape)

rows, cols = grid_shape.shape
print("Initial Grid:")
for i in range(rows):
    row_bytes = [grid.get_value((i, j)) for j in range(cols)]
    hex_bytes = " ".join(f"{b:02x}" for b in row_bytes)
    print(f"  row {i:2d}: {hex_bytes}")

Initial Grid:
  row  0: 49 80 c5 6a ff 70 08 84 9f 15 2c 48
  row  1: e3 94 73 e2 5b a3 70 06 07 6a 21 ca
  row  2: d6 73 44 27 b2 47 0d 19 a7 b7 bb 50
  row  3: 76 ee 44 bb bb 75 ec 41 19 d0 b2 e5
  row  4: c3 a4 fd 46 95 70 84 7c 42 f1 83 e6
  row  5: cd e3 96 3c 40 53 f2 c3 fb f3 35 67
  row  6: 53 2a 8b e6 c2 50 c1 fe 59 38 ce 09
  row  7: 8e 6d 50 a0 ac 33 ea 50 b8 c4 2f bb
  row  8: 1d 69 bf 11 97 b0 d7 10 d0 7e 71 59
  row  9: ed cc ba 6e 18 d9 14 81 b0 22 3f 2a
  row 10: 5b 56 ca 6c 18 ec a6 bc 7b 55 ee 0a
  row 11: 9f 4f f1 09 02 72 93 a7 40 7b 1d 2b


Keygen API

returns Secret key := stencil_lib.SecretKey type

In [37]:
secret_key = stencil_lib.keygen(
    stencil_cfg,
    cipher_cfg=cipher_cfg,
)

print("Secret Key")
print(f"  Cipher    : {secret_key.cipher_cfg.algo}")
print(f"  Partitions: {secret_key.partition_list}  ({len(secret_key.partition_list)} total, sum={sum(secret_key.partition_list)} bytes)")
print(f"  Stencils  : {len(secret_key.stencils)}")
for i, s in enumerate(secret_key.stencils):
    coords_str = ", ".join(f"({r},{c})" for r, c in s.coords)
    print(f"    [{i}] shape={s.shape!r}  len={s.len}  coords=[{coords_str}]")


Secret Key
  Cipher    : vigenere
  Partitions: [9, 4, 1, 2]  (4 total, sum=16 bytes)
  Stencils  : 4
    [0] shape='skewconnected'  len=9  coords=[(3,7), (2,6), (2,5), (1,5), (1,6), (0,6), (0,7), (1,8), (1,7)]
    [1] shape='skewconnected'  len=4  coords=[(6,3), (5,4), (6,4), (7,5)]
    [2] shape='skewconnected'  len=1  coords=[(9,11)]
    [3] shape='skewconnected'  len=2  coords=[(2,2), (3,1)]


Encrypt API 

returns obfuscated_grid := stencil_lib.Grid type

In [38]:
obfuscated_grid = stencil_lib.encrypt(plaintext, secret_key, stencil_cfg, grid)
print("\nWith Encryption API call, (Cipher text embedded) Obfuscated Grid is ready")



With Encryption API call, (Cipher text embedded) Obfuscated Grid is ready


Print Cipher text and Obfuscated grid for Demo purpose:

In [39]:
from IPython.display import display, HTML

# Just for Demo purpose
ciphertext = cipher_cfg.encrypt(plaintext)

print(f"Ciphertext: {' '.join(f'{b:02x}' for b in ciphertext)}")

PART_COLORS = ["#f0a500", "#4fc3f7", "#81c784", "#f06292", "#ce93d8", "#80cbc4"]

# --- Partitioned ciphertext ---
parts_lines = []
offset = 0
for i, length in enumerate(secret_key.partition_list):
    chunk = ciphertext[offset: offset + length]
    color = PART_COLORS[i % len(PART_COLORS)]
    hex_part = " ".join(f"{b:02x}" for b in chunk)
    parts_lines.append(
        f'  P{i} ({length:2d}B): <span style="color:{color};font-weight:bold">{hex_part}</span>'
    )
    offset += length

display(HTML(
    "<b>Ciphertext — by partition</b>"
    '<pre style="line-height:1.8">' + "\n".join(parts_lines) + "</pre>"
))

# --- Obfuscated grid: all bytes of a partition share its color; first byte underlined ---
coord_to_part  = {(r, c): i for i, s in enumerate(secret_key.stencils) for r, c in s.coords}
first_coords   = {s.coords[0] for s in secret_key.stencils}

rows_html = []
rows, cols = stencil_cfg.grid_shape.shape
for i in range(rows):
    cells = []
    for j in range(cols):
        b = obfuscated_grid.get_value((i, j))
        token = f"{b:02x}"
        if (i, j) in coord_to_part:
            part_idx = coord_to_part[(i, j)]
            color = PART_COLORS[part_idx % len(PART_COLORS)]
            underline = ";text-decoration:underline" if (i, j) in first_coords else ""
            token = f'<span style="color:{color};font-weight:bold{underline}">{token}</span>'
        cells.append(token)
    rows_html.append(f'  row {i:2d}: {' '.join(cells)}')

legend = "  ".join(
    f'<span style="color:{PART_COLORS[i % len(PART_COLORS)]};font-weight:bold">P{i}</span>'
    for i in range(len(secret_key.stencils))
)
display(HTML(
    f"<b>Obfuscated Grid</b> — {legend} (underlined = partition start)<br>"
    '<pre style="line-height:1.6">' + "\n".join(rows_html) + "</pre>"
))


Ciphertext: 96 4e 4d 27 8e 08 c0 8d 50 ad 78 f1 ac 2d bd 63


Decryprt API 

returns plaintext := bytes type

In [41]:
recovered = stencil_lib.decrypt(obfuscated_grid, secret_key, stencil_cfg)
assert recovered == plaintext

print(f"Decrypted Plaintext : {' '.join(f'{b:02x}' for b in recovered)}")

Decrypted Plaintext : 4b 09 f4 dc 49 af 75 48 f7 62 33 98 61 e8 64 18
